# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Prep/Setup

In [20]:
import duckdb
from getpass import getpass
import pandas as pd
import numpy as np

con = duckdb.connect()
hf_token = getpass("Paste your Hugging Face READ token: ")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{base}/fact_content_daily_performance/month=2026-03/data_0.parquet"

# Label — identical to w04/w05/w06
label_df = con.sql(f"""
    WITH halves AS (
        SELECT
            content_hash_id,
            SUM(CASE WHEN report_date < '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
            SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_second_half
        FROM read_parquet('{month_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        content_hash_id,
        CASE WHEN impr_second_half < impr_first_half THEN 1 ELSE 0 END AS is_declining_proxy
    FROM halves
    WHERE impr_first_half > 0
""").df()

# Full-month position/click signal — for the baseline rule's score + tie-break only
pf = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_full,
        SUM(gsc_impressions) AS total_impressions_full,
        SUM(gsc_clicks) AS total_clicks_full
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
pf_valid = pf.dropna(subset=["avg_position_full"]).copy()
pf_valid["eligible"] = pf_valid["total_impressions_full"] >= 10
pf_valid["zero_clicks_at_position"] = (
    (pf_valid["avg_position_full"] <= 10) & (pf_valid["total_clicks_full"] == 0) & (pf_valid["eligible"])
).astype(int)

# First-half-only features — what the models actually train on
pf_fh = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_fh,
        SUM(gsc_impressions) AS total_impressions_fh,
        SUM(gsc_clicks) AS total_clicks_fh
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND report_date < '2026-03-16'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
pf_fh = pf_fh.dropna(subset=["avg_position_fh"]).copy()
pf_fh["ctr_fh"] = pf_fh["total_clicks_fh"] / pf_fh["total_impressions_fh"]

# Leakage-safe position trend, days 1-15 only
postrend = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN report_date < '2026-03-08' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk1,
        AVG(CASE WHEN report_date >= '2026-03-08' AND report_date < '2026-03-16' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk2
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND report_date < '2026-03-16'
    GROUP BY content_hash_id
""").df()
postrend = postrend.dropna(subset=["avg_position_wk1", "avg_position_wk2"])
postrend["position_change"] = postrend["avg_position_wk2"] - postrend["avg_position_wk1"]
postrend["position_worsened"] = (postrend["position_change"] > 0).astype(int)
postrend_eligible = postrend.merge(pf_valid[["content_hash_id", "eligible"]], on="content_hash_id", how="left")
postrend_eligible = postrend_eligible[postrend_eligible["eligible"] == True].copy()

# Client map for grouping
client_map = con.sql(f"""
    SELECT DISTINCT content_hash_id, client_hash_id
    FROM read_parquet('{month_path}')
""").df()

# Assemble model_df — same joins, same order, as Week 5/6
model_df = pf_fh.merge(
    pf_valid[["content_hash_id", "total_impressions_full", "eligible", "zero_clicks_at_position"]],
    on="content_hash_id", how="inner"
)
model_df = model_df.merge(
    postrend_eligible[["content_hash_id", "position_change", "position_worsened"]],
    on="content_hash_id", how="left"
)
model_df["has_position_trend"] = model_df["position_change"].notna().astype(int)
model_df["position_change"] = model_df["position_change"].fillna(0)
model_df["position_worsened"] = model_df["position_worsened"].fillna(0).astype(int)
model_df = model_df.merge(client_map, on="content_hash_id", how="left").dropna(subset=["client_hash_id"])
model_df = model_df.merge(label_df, on="content_hash_id", how="inner").sort_values("content_hash_id").reset_index(drop=True)

model_df["log_impressions_fh"] = np.log1p(model_df["total_impressions_fh"])
model_df["log_clicks_fh"] = np.log1p(model_df["total_clicks_fh"])

feature_cols = ["avg_position_fh", "log_impressions_fh", "log_clicks_fh", "ctr_fh",
                "position_change", "has_position_trend"]

model_df["baseline_score"] = model_df["zero_clicks_at_position"] * 2 + model_df["position_worsened"]

print(f"model_df: {model_df.shape}, base rate: {model_df['is_declining_proxy'].mean():.3f}")

# --- Out-of-fold RF scoring: 5-fold GroupKFold, same seed as w05/w06 ---
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier

def make_rf():
    return RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1)

X = model_df[feature_cols].astype(float)
y = model_df["is_declining_proxy"].astype(int)
groups = model_df["client_hash_id"]

oof_rf = np.full(len(model_df), np.nan)
gkf = GroupKFold(n_splits=5)
for fold_num, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    rf = make_rf()
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_rf[test_idx] = rf.predict_proba(X.iloc[test_idx])[:, 1]

model_df["oof_rf_score"] = oof_rf
print(f"OOF coverage: {model_df['oof_rf_score'].notna().sum()} / {len(model_df)} rows")

Paste your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

model_df: (150675, 16), base rate: 0.438
OOF coverage: 150675 / 150675 rows


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [25]:
# Thresholds — data-driven, not arbitrary
high_score_cut = model_df.loc[model_df["baseline_score"] == 0, "oof_rf_score"].quantile(0.90)
large_swing_cut = model_df["position_change"].abs().quantile(0.90)

def assign_archetype(row):
    if row["zero_clicks_at_position"] == 1 and row["position_worsened"] == 1:
        return 1, "zero_clicks_and_worsened", "Refresh content + overhaul title/meta"
    if row["zero_clicks_at_position"] == 1:
        return 2, "zero_clicks_only", "Overhaul title & meta"
    if row["position_worsened"] == 1:
        return 3, "position_worsened_only", "Refresh content"
    if row["baseline_score"] == 0 and row["oof_rf_score"] >= high_score_cut:
        caution = abs(row["position_change"]) >= large_swing_cut
        action = "Flag for manual review (model-only signal — verify before acting)" if caution \
                 else "Flag for manual review"
        return 4, "model_only_catch", action
    return 5, "no_flag", "Monitor"

archetype_info = model_df.apply(assign_archetype, axis=1, result_type="expand")
model_df[["priority_tier", "archetype", "action"]] = archetype_info

# Combined score: model score does the ordering, rule adds a bounded nudge —
# neither signal fully vetoes the other
model_df["combined_score"] = model_df["oof_rf_score"] + 0.03 * model_df["baseline_score"]

ranked_queue = model_df.sort_values("combined_score", ascending=False).reset_index(drop=True)

print(ranked_queue["archetype"].value_counts())
print(ranked_queue[["content_hash_id", "archetype", "action", "baseline_score",
                     "oof_rf_score", "combined_score"]].head(20))
caution_count = (
    (ranked_queue["archetype"] == "model_only_catch") &
    (ranked_queue["position_change"].abs() >= large_swing_cut)
).sum()
print(f"\nmodel_only_catch rows with caution flag: {caution_count} / {(ranked_queue['archetype']=='model_only_catch').sum()}")

archetype
no_flag                     65567
position_worsened_only      52753
zero_clicks_only            14457
zero_clicks_and_worsened    10609
model_only_catch             7289
Name: count, dtype: int64
             content_hash_id               archetype                  action  \
0   content_403a36188e13fcc8  position_worsened_only         Refresh content   
1   content_2435b8bb25eeebd9  position_worsened_only         Refresh content   
2   content_df977de3b77ec57c  position_worsened_only         Refresh content   
3   content_b5a91be0a10cd899  position_worsened_only         Refresh content   
4   content_4e48bd81bb37eb4f  position_worsened_only         Refresh content   
5   content_5effb301ded55c21  position_worsened_only         Refresh content   
6   content_334bcb2761d0f9c7        model_only_catch  Flag for manual review   
7   content_83a700e06cf9e676        model_only_catch  Flag for manual review   
8   content_bab284527da6960a        model_only_catch  Flag for manual revi

The queue ranks all 150,675 scored pages by oof_rf_score (Week 5/6's random forest, five-fold GroupKFold, random_state=42) plus a small bonus from the Week 4 rule (0.03 × baseline_score, capped at 0.09), with the model doing nearly all the ordering and the rule only breaking close ties. Archetype and action come from the rule's flags alone, so reason and rank are answered separately, not by the same number.

There are five archetypes chosen, those being; zero_clicks_and_worsened (10,609 — refresh + overhaul), zero_clicks_only (14,457 — overhaul title/meta), position_worsened_only (52,753 — refresh), model_only_catch (7,289 — unflagged pages in the model's top 10%, sent to manual review), and no_flag (65,567 — monitor).

There is a caution flag this archetype was built to carry, that being the previously documented RF failure mode of high score driven by a large position swing rather than real decline. It was found to fire on only 42 of 7,289 rows (0.6%), marking it as a small minority needing a second look, not grounds to distrust the archetype.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This queue is built for a content editor or SEO strategist doing weekly or monthly triage — it tells them where to look first, not what to conclude. Every action is intended as a starting point for review, not an instruction to execute unread.

This whole system stops being valid past a few specific edges. The pipeline itself — feature construction, the GroupKFold fitting approach — is month-agnostic and can be rerun on any month's data, but the scores in this queue come from models trained only on March 2026. Nothing here was tested on another month, so a new month means retraining from scratch, not reapplying these scores to new pages.

The model was never given a direct zero-click signal (zero_clicks_at_position_fh was built but only used in a Week 6 diagnostic, never added to feature_cols), so it only picks up CTR problems indirectly. That's why zero-click archetypes carry the largest rule bonus in Section 1 but rarely reach the top of the blended queue — it's not that the rule is wrong, but rather the model isn't built to weigh that signal directly.

This queue can't detect a page that gets clicks but loses people immediately — the one candidate signal for that, session_rate, was tested in Week 4 and rejected because it moved in the wrong direction, confounded with impression volume rather than measuring real engagement. This is a gap in what the present signals measure, not something the model missed.

oof_rf_score is pooled from five separately trained fold models, and Week 6 found they aren't fully comparable — one fold's model produced systematically higher scores without being more accurate. It is best advised to treat close scores across the full population with some caution; a 0.02 gap between two rows isn't necessarily meaningful.

Confidence in the model's ordering is strong at K=50 and up, backed by a bootstrap CI clearly above zero, but soft at K=20, where RF only won 3 of 5 folds. Therefore the very top of the queue should be subject to more scrutiny than the model's confidence alone would suggest.

Finally, this queue recommends refresh candidates based on decline signals — it does not show or guarantee that refreshing causes recovery: acting on this queue and later seeing improvement doesn't directly confirm that the action caused it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.